# Exercise 6 v2 - Content-Based Car Recommendation System

Goal: given one specific car, return a ranked list of similar cars.

This version keeps the original `Task_6.ipynb` untouched and improves the recommender by:

- cleaning missing values before feature engineering,
- using exact `row_id` inputs so repeated model names are not ambiguous,
- splitting `Market Category` into multi-label features,
- applying feature weights so vehicle type/specs matter more than brand alone,
- evaluating recommendations against a random baseline.

In [ ]:
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path('data.csv')
df_raw = pd.read_csv(DATA_PATH)
df_raw = df_raw.reset_index().rename(columns={'index': 'row_id'})

print(df_raw.shape)
display(df_raw.head())

## 1. Data Cleaning

The dataset contains car specifications such as horsepower, MPG, MSRP, body style, size, drivetrain, and fuel type. Missing numeric values are filled with medians. Missing categorical values are filled with `Unknown`.

In [ ]:
df = df_raw.copy()

numeric_cols = [
    'Year', 'Engine HP', 'Engine Cylinders', 'Number of Doors',
    'highway MPG', 'city mpg', 'Popularity', 'MSRP'
]

categorical_cols = [
    'Make', 'Engine Fuel Type', 'Transmission Type', 'Driven_Wheels',
    'Vehicle Size', 'Vehicle Style'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    df[col] = df[col].fillna('Unknown').astype(str)

df['Market Category'] = df['Market Category'].fillna('Unknown').astype(str)

df['car_label'] = df.apply(
    lambda row: f"{row['row_id']}: {row['Make']} {row['Model']} ({row['Year']})",
    axis=1
)

print('Missing values after cleaning:', int(df.isna().sum().sum()))
display(df[['row_id', 'car_label', 'Vehicle Style', 'Vehicle Size', 'Engine HP', 'highway MPG', 'city mpg', 'MSRP']].head(10))

## 2. Feature Engineering

A content-based recommender needs features that describe the car itself. I use:

- numeric specs: year, horsepower, cylinders, doors, MPG, popularity, MSRP,
- categorical specs: fuel type, transmission, drivetrain, size, body style,
- multi-label market categories: luxury, performance, crossover, etc.,
- make/brand with a low weight so the system can recommend similar competitors, not only same-brand cars.

In [ ]:
def build_feature_matrix(df):
    numeric = df[numeric_cols].copy()
    numeric_scaled = pd.DataFrame(
        StandardScaler().fit_transform(numeric),
        columns=numeric_cols,
        index=df.index,
    )

    # Price and horsepower can be very skewed. A log version helps compare realistic ranges.
    numeric_scaled['log_MSRP'] = StandardScaler().fit_transform(np.log1p(df[['MSRP']]))[:, 0]
    numeric_scaled['log_Engine_HP'] = StandardScaler().fit_transform(np.log1p(df[['Engine HP']]))[:, 0]

    make_features = pd.get_dummies(df['Make'], prefix='make', dtype=float)
    spec_categories = pd.get_dummies(
        df[['Engine Fuel Type', 'Transmission Type', 'Driven_Wheels', 'Vehicle Size', 'Vehicle Style']],
        dtype=float,
    )

    market_tokens = (
        df['Market Category']
        .str.get_dummies(sep=',')
        .rename(columns=lambda name: f"market_{name.strip().replace(' ', '_')}")
        .astype(float)
    )

    feature_parts = {
        'numeric': numeric_scaled,
        'make': make_features,
        'spec_categories': spec_categories,
        'market': market_tokens,
    }

    # Weights encode what should matter most for a human-sensible car recommendation.
    weights = {
        'numeric': 1.25,
        'make': 0.25,
        'spec_categories': 1.40,
        'market': 1.10,
    }

    weighted_parts = []
    for group_name, part in feature_parts.items():
        weighted_parts.append(part * weights[group_name])

    features = pd.concat(weighted_parts, axis=1).fillna(0.0)
    return features, weights

features, feature_weights = build_feature_matrix(df)
similarity_matrix = cosine_similarity(features)

print('Feature matrix shape:', features.shape)
print('Feature weights:', feature_weights)

## 3. Recommendation Functions

`row_id` is the safest input because the same model name appears many times with different trims, years, engines, and prices.

In [ ]:
display_cols = [
    'row_id', 'Make', 'Model', 'Year', 'Vehicle Style', 'Vehicle Size',
    'Engine Fuel Type', 'Transmission Type', 'Driven_Wheels',
    'Engine HP', 'Engine Cylinders', 'highway MPG', 'city mpg', 'MSRP'
]

def find_cars(model=None, make=None, year=None, limit=15):
    matches = pd.Series(True, index=df.index)
    exact_model = pd.Series(False, index=df.index)
    if model is not None:
        model_text = df['Model'].fillna('').astype(str)
        exact_model = model_text.str.lower().eq(str(model).lower())
        matches &= exact_model | model_text.str.contains(model, case=False, na=False)
    if make is not None:
        matches &= df['Make'].str.contains(make, case=False, na=False)
    if year is not None:
        matches &= df['Year'].eq(year)

    result = df.loc[matches, ['car_label'] + display_cols].copy()
    if model is not None and not result.empty:
        result['_exact_model'] = result['Model'].str.lower().eq(str(model).lower())
        result = result.sort_values(['_exact_model', 'Year', 'MSRP'], ascending=[False, False, True])
        result = result.drop(columns=['_exact_model'])
    return result.head(limit)

def recommend_by_row_id(row_id, top_n=10, unique_models=True):
    if row_id not in set(df['row_id']):
        raise ValueError(f'Unknown row_id: {row_id}')

    target_idx = int(df.index[df['row_id'].eq(row_id)][0])
    target = df.loc[target_idx]
    scores = list(enumerate(similarity_matrix[target_idx]))
    scores = sorted(scores, key=lambda item: item[1], reverse=True)

    selected = []
    seen_model_keys = set()
    for candidate_idx, score in scores:
        if candidate_idx == target_idx:
            continue

        candidate = df.loc[candidate_idx]
        model_key = (candidate['Make'], candidate['Model'])
        target_key = (target['Make'], target['Model'])

        if unique_models and model_key == target_key:
            continue
        if unique_models and model_key in seen_model_keys:
            continue

        seen_model_keys.add(model_key)
        selected.append((candidate_idx, score))
        if len(selected) == top_n:
            break

    result = df.loc[[idx for idx, _ in selected], display_cols].copy()
    result.insert(0, 'similarity', [score for _, score in selected])
    result.insert(0, 'input_car', target['car_label'])
    return result.reset_index(drop=True)

def recommend_by_search(model, make=None, year=None, top_n=10):
    matches = find_cars(model=model, make=make, year=year, limit=20)
    if matches.empty:
        raise ValueError('No matching car found. Try a broader search.')
    row_id = int(matches.iloc[0]['row_id'])
    print('Using input car:', matches.iloc[0]['car_label'])
    return recommend_by_row_id(row_id, top_n=top_n)

display(find_cars(model='Corolla', make='Toyota', limit=5))

## 4. Example Recommendations

In [ ]:
examples = [
    ('Odyssey', 'Honda'),
    ('Corolla', 'Toyota'),
    ('Mustang', 'Ford'),
    ('F-150', 'Ford'),
    ('911', 'Porsche'),
]

for model, make in examples:
    print('\n' + '=' * 90)
    print(f'Recommendations for {make} {model}')
    display(recommend_by_search(model=model, make=make, top_n=7))

## 5. Evaluation

Recommendation systems often lack explicit labels for what is correct. Here I evaluate with proxy metrics:

- same vehicle style rate,
- same vehicle size rate,
- same fuel type rate,
- average normalized specification distance,
- comparison with a random recommender baseline.

A good content-based recommender should produce higher style/size/fuel matches and lower specification distance than random recommendations.

In [ ]:
eval_numeric_cols = ['Engine HP', 'Engine Cylinders', 'highway MPG', 'city mpg', 'MSRP']
numeric_ranges = df[eval_numeric_cols].max() - df[eval_numeric_cols].min()
numeric_ranges = numeric_ranges.replace(0, 1)

def evaluate_rows(row_ids, top_n=10, random_seed=42):
    rng = random.Random(random_seed)
    rows = []

    for row_id in row_ids:
        target = df.loc[df['row_id'].eq(row_id)].iloc[0]
        recs = recommend_by_row_id(int(row_id), top_n=top_n)

        candidate_pool = df[df['row_id'].ne(row_id)].sample(n=top_n, random_state=rng.randint(1, 10_000))

        for method, candidates in [('content_based', recs), ('random_baseline', candidate_pool)]:
            spec_distance = (
                (candidates[eval_numeric_cols].reset_index(drop=True) - target[eval_numeric_cols].astype(float))
                .abs()
                .div(numeric_ranges, axis=1)
                .mean(axis=1)
                .mean()
            )

            rows.append({
                'input_car': target['car_label'],
                'method': method,
                'same_style_rate': (candidates['Vehicle Style'].values == target['Vehicle Style']).mean(),
                'same_size_rate': (candidates['Vehicle Size'].values == target['Vehicle Size']).mean(),
                'same_fuel_type_rate': (candidates['Engine Fuel Type'].values == target['Engine Fuel Type']).mean(),
                'avg_normalized_spec_distance': spec_distance,
                'avg_similarity': candidates['similarity'].mean() if 'similarity' in candidates.columns else np.nan,
                'unique_make_count': candidates['Make'].nunique(),
            })

    return pd.DataFrame(rows)

test_row_ids = [
    int(find_cars('Odyssey', make='Honda', limit=1).iloc[0]['row_id']),
    int(find_cars('Corolla', make='Toyota', limit=1).iloc[0]['row_id']),
    int(find_cars('Mustang', make='Ford', limit=1).iloc[0]['row_id']),
    int(find_cars('F-150', make='Ford', limit=1).iloc[0]['row_id']),
    int(find_cars('911', make='Porsche', limit=1).iloc[0]['row_id']),
]

evaluation = evaluate_rows(test_row_ids, top_n=10)
display(evaluation)

summary = evaluation.groupby('method').mean(numeric_only=True).sort_index()
display(summary)

## 6. Interpretation

The recommender is content-based because it does not use user ratings or purchase history. It compares cars by their own attributes. The evaluation compares it with random recommendations. If the content-based row has better style/size/fuel match rates and lower normalized spec distance, the recommender is behaving sensibly.